# Week 2, Day 4 — Class Imbalance: Weighting, SMOTE, and a Real Leakage Trap

Day 1 showed that accuracy can be misleading. Today pushes that to its most extreme form: on a dataset where the thing you're trying to catch is rare, a model can hit 97%+ accuracy while catching **none** of it, just by leaning on the majority class. Fixing that requires the model to actually pay attention to the minority class during training (**class weighting**), or requires giving it more minority examples to learn from (**SMOTE**) — two different mechanisms, not a strict better/worse pair.

The dataset here is synthetic (`make_classification`, 10,000 rows, ~2.4% positive) rather than a further-subsampled Titanic. Days 1–3 already showed what happens when a small dataset's positive class gets pushed too hard — today needs enough real positive examples in the test set to trust a precision/recall comparison, which a shrunk Titanic slice can't reliably give.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X, y = make_classification(
    n_samples=10000, weights=[0.98, 0.02], flip_y=0.01, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"overall positives: {y.sum()} / {len(y)} ({y.mean():.2%})")
print(f"train positives:   {y_train.sum()} / {len(y_train)} ({y_train.mean():.2%})")
print(f"test positives:    {y_test.sum()} / {len(y_test)} ({y_test.mean():.2%})")


def report(name, y_true, y_pred):
    print(
        f"{name:28s} acc={accuracy_score(y_true, y_pred):.4f}  "
        f"prec={precision_score(y_true, y_pred, zero_division=0):.4f}  "
        f"rec={recall_score(y_true, y_pred, zero_division=0):.4f}  "
        f"f1={f1_score(y_true, y_pred, zero_division=0):.4f}"
    )

## Block 1 — The problem, no handling at all

Fit a plain `RandomForestClassifier` with no imbalance handling — no class weights, no resampling — and compare it against a baseline that doesn't even look at the features, just predicts "negative" every time.

In [ ]:
rf_plain = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_plain.fit(X_train, y_train)

report("plain RandomForest", y_test, rf_plain.predict(X_test))
report("always predict majority", y_test, np.zeros_like(y_test))

**98.0% accuracy, but recall is 0.2041** — the model catches only 10 of the 49 real positives in the test set. It's not *literally* as blind as the always-negative baseline (which gets 0 recall by definition), but 98% accuracy next to "misses 4 out of 5 positives" is arguably the more dangerous failure mode: a model with a little bit of signal still reads as "working fine" on any dashboard that only tracks accuracy, right up until someone checks recall specifically. This is Day 1's lesson made concrete: accuracy alone can't tell these two models apart in any way that matters.

## Block 2 — Class weighting

`class_weight='balanced'` reweights the training loss inversely proportional to class frequency — misclassifying a rare positive now costs roughly as much as misclassifying the ~40 negatives it's outnumbered by. The model stops getting a free pass for ignoring the minority class; nothing about the data changes, only how errors on it are penalized during training.

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=200, max_depth=6, class_weight="balanced", random_state=42
)
rf_balanced.fit(X_train, y_train)

report("class_weight='balanced'", y_test, rf_balanced.predict(X_test))

Accuracy drops nine points (0.9800 → 0.9235) — if you only checked accuracy, you'd conclude this made the model worse. Recall climbs from 0.2041 to **0.6735** (33 of 49 positives caught, up from 10), and F1 goes from 0.3333 to 0.3014. F1 actually dips slightly here even though recall improved a lot, because precision fell hard (0.9091 → 0.1941) — weighting doesn't give recall for free, it trades away some precision to get it. Day 1's lesson made operational, not just observed.

## Block 3 — SMOTE, applied correctly

SMOTE (Synthetic Minority Over-sampling Technique) generates new minority-class examples by interpolating between real minority points — it doesn't duplicate existing rows, it manufactures plausible new ones along the line between a minority point and its nearest minority neighbors.

**Critical rule, stated before touching the code:** SMOTE gets fit on the training set only, *after* the split — never on the full dataset before splitting. Block 4 shows exactly why, but the short version is that a synthetic point can end up looking like a near-clone of a real point across the train/test boundary if the split happens after resampling.

In [ ]:
from imblearn.over_sampling import SMOTE

X_train_smote, y_train_smote = SMOTE(random_state=42).fit_resample(X_train, y_train)
print(
    f"SMOTE-resampled train: {y_train_smote.sum()} positives / {len(y_train_smote)} "
    f"({y_train_smote.mean():.2%}), test set untouched"
)

rf_smote = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)

report("SMOTE (correct order)", y_test, rf_smote.predict(X_test))

Train set goes from 194 positives / 8000 (2.4%) to a balanced 7806 / 15612 (50%) — SMOTE oversamples the minority up to match the majority count, by default. Test set stays exactly as it was (49 positives / 2000), because it was never touched.

The genuinely interesting result: SMOTE catches **more** real positives than class weighting (36 vs. 33 out of 49 — recall 0.7347 vs. 0.6735), but precision (0.1935) lands almost exactly where class weighting's did (0.1941), so F1 (0.3064) ends up nearly tied with weighting's (0.3014) rather than clearly ahead or behind. Neither technique dominates here — they reach nearly the same tradeoff point through different mechanisms (reweighting errors vs. manufacturing more minority examples), and which you'd actually pick depends on Day 3's cost framework: if a missed positive is far more expensive than a false alarm, SMOTE's extra recall is worth having even though the F1s are a wash.

## Block 4 — The leakage trap

Now the mistake, run on purpose. `SMOTE().fit_resample(X, y)` **before** `train_test_split` is one line of reordering from Block 3 — and it's a natural-looking bug, the kind written without a second thought when SMOTE gets bolted onto an existing pipeline near the top instead of after the split.

In [ ]:
# WRONG ORDER: resampling before the split
X_full_smote, y_full_smote = SMOTE(random_state=42).fit_resample(X, y)
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_full_smote, y_full_smote, test_size=0.2, random_state=42, stratify=y_full_smote
)
print(
    f"leaky resampled full set: {y_full_smote.sum()} positives / {len(y_full_smote)} "
    f"-> leaky test set: {y_test_leak.sum()} positives / {len(y_test_leak)}"
)

rf_leak = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_leak.fit(X_train_leak, y_train_leak)

report("SMOTE (LEAKY order)", y_test_leak, rf_leak.predict(X_test_leak))

F1 jumps from an honest ~0.31 (Block 3) to **0.8599** — nearly 3x higher. Notice the leaky test set isn't even the same test set anymore: because SMOTE ran on the full data first, splitting afterward divides up 19514 balanced rows instead of the original 10000, so the leaky test set is 3903 rows that are ~50% positive (1951 positives), not the original 49-positive, 2.45%-positive test set. This alone should be a red flag — the model isn't being measured on the same problem anymore.

Now the actual proof of *why* the number is fiction, not just an assertion:

In [ ]:
from sklearn.neighbors import NearestNeighbors

# imblearn appends synthetic rows after the real ones, so index >= len(X) marks a synthetic point
is_synthetic = np.zeros(len(X_full_smote), dtype=bool)
is_synthetic[len(X) :] = True

idx_all = np.arange(len(X_full_smote))
idx_train, idx_test = train_test_split(
    idx_all, test_size=0.2, random_state=42, stratify=y_full_smote
)

synth_in_test = is_synthetic[idx_test]
print(
    f"synthetic points that landed in the 'test' set: {synth_in_test.sum()} / {len(idx_test)}"
)

nn = NearestNeighbors(n_neighbors=1).fit(X_full_smote[idx_train])

dist_synth, _ = nn.kneighbors(X_full_smote[idx_test[synth_in_test]])
dist_real, _ = nn.kneighbors(X_full_smote[idx_test[~synth_in_test]])

print(
    f"median distance, synthetic-in-test -> nearest train point:  {np.median(dist_synth):.4f}"
)
print(
    f"median distance, real-in-test -> nearest train point:       {np.median(dist_real):.4f}"
)
print(
    f"fraction of synthetic test points within 0.1 of a train point: {(dist_synth < 0.1).mean():.2%}"
)
print(
    f"fraction of real test points within 0.1 of a train point:      {(dist_real < 0.1).mean():.2%}"
)

1904 of the 3903 leaky test rows are synthetic — nearly half the "held-out" set is fabricated data. And those synthetic test points sit a **median distance of 0.147** from their nearest training point, versus **2.888** for genuinely real, non-leaked test points — roughly 20x closer. 37% of synthetic test points sit within 0.1 of a training point, versus 1.65% of real ones. That's the mechanism made concrete: SMOTE interpolates between neighbors, so a synthetic point that lands in "test" is often a near-clone of the real training points it was interpolated from. The test set isn't measuring generalization anymore — it's measuring how well the model memorized points it was functionally shown during training.

This is Week 1's leakage lesson again, but in a far more realistic, far more common form than a deliberately-contrived example: a one-line reordering that looks completely innocent, the kind of mistake real practitioners actually make when SMOTE gets added to a pipeline near the top instead of right after the split.

## Block 5 — Wrap-up

Four results, side by side:

| approach | acc | prec | rec | f1 |
|---|---|---|---|---|
| baseline (no handling) | 0.9800 | 0.9091 | 0.2041 | **0.3333** |
| class weighting | 0.9235 | 0.1941 | 0.6735 | **0.3014** |
| SMOTE (correct order) | 0.9185 | 0.1935 | 0.7347 | **0.3064** |
| SMOTE (**leaky** order) | 0.8675 | 0.9125 | 0.8129 | **0.8599** |

Notice the baseline's F1 isn't even the lowest of the three honest results — it wasn't literally blind here, just bad at the one thing that mattered (0.2041 recall). The real lesson isn't a clean monotonic ordering; it's that weighting and SMOTE land at nearly the same honest tradeoff point through different mechanisms, while getting the *ordering* of SMOTE relative to the train/test split wrong doesn't degrade the result gently — it manufactures a completely fictional one, nearly 3x higher, on a test set that has quietly stopped measuring what it claims to measure.

**One-sentence takeaway:** handling imbalance well requires getting two separate things right — choosing a technique (class weighting and SMOTE are different tradeoffs, not a strict better/worse pair) and ordering it correctly relative to your train/test split, where a single line out of place doesn't degrade your numbers gently, it manufactures a completely fictional result.